# Fine-Tuning IndoBERT Transformer untuk Analisis Kepercayaan UMKM Kosmetik Halal
**Capstone Design Project**: Explainable AI & NLP Kosmetik Halal

Notebook ini digunakan untuk melatih model Transformer `indobenchmark/indobert-base-p1` pada GPU Google Colab.

In [ ]:
# 1. Install Library Dependencies
!pip install -q transformers torch datasets shap accelerate pandas scikit-learn joblib

In [ ]:
# 2. Import Module & Inisialisasi Device GPU
import os
import json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# 3. Load Dataset Training & Validation
train_df = pd.read_csv('data/annotated/train.csv')
val_df = pd.read_csv('data/annotated/val.csv')
test_df = pd.read_csv('data/annotated/test.csv')

print(f'Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}')

In [ ]:
# 4. Inisialisasi Tokenizer & PyTorch Dataset Class
MODEL_NAME = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HalalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, item):
        text = str(self.texts[item])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

In [ ]:
# 5. Training Loop IndoBERT Model
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

print('Prosedur training siap dieksekusi di Colab GPU!')